# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AmaanJilani1/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

*Lane: Refresh / Content Opportunity Scoring. Label: page impressions fell more than 20% (last 30d vs prior 30d). Language: observed, measured, directional, decision-support.*

## 1. Method choice and why

Two models, one question: *which pages will see impressions fall >20% next?*

- **Logistic regression** (scaled, median-imputed): readable coefficients, so each recommendation can carry a reason a human trusts.
- **Gradient boosting** (`HistGradientBoostingClassifier`, shallow, 200 rounds): can pick up non-linear patterns; used to check whether complexity buys anything.

Both compare against the two baselines from ML-07. Features are past-window only (days 31-90) plus static content/keyword properties.

In [ ]:

import pandas as pd, numpy as np, os, warnings
warnings.filterwarnings("ignore")
URL = "https://raw.githubusercontent.com/AmaanJilani1/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(URL)

# TARGET only. trend_direction is the label source -> never a feature.
df["y"] = (df["trend_direction"] == "down").astype(int)

# Clean past windows. The 90d total = days 1-30 (last30) + 31-60 (prev30) + 61-90 (old).
# Subtracting gives a window that is strictly BEFORE the label window.
for a, b in [("imp", "impressions"), ("clk", "clicks"), ("ses", "sessions")]:
    df[a + "_old"]  = df[b + "_90d"] - df[b + "_last_30d"] - df[b + "_prev_30d"]   # days 61-90 ago
    df[a + "_prev"] = df[b + "_prev_30d"]                                          # days 31-60 ago

# Eligibility: a page can only "decline" if it had visibility to lose.
FLOOR = 100
el = df[df["imp_prev"] >= FLOOR].copy().reset_index(drop=True)
print(f"all pages: {len(df):,} | eligible (prev30 impressions >= {FLOOR}): {len(el):,} "
      f"| clients: {el.client_id.nunique()} | decline base rate: {el.y.mean():.3f}")


from sklearn.ensemble import HistGradientBoostingClassifier as HGB
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# Raw columns the features are built from. Everything is from BEFORE the label window.
SOURCES = ["imp_old","imp_prev","clk_old","clk_prev","ses_old","ses_prev","search_volume","competition",
           "cpc","word_count","content_age_days","days_since_last_update","content_type","main_intent"]
FORBIDDEN = {"trend_direction","trend_pct","impressions_last_30d","clicks_last_30d","sessions_last_30d",
             "impressions_90d","clicks_90d","sessions_90d","ctr","avg_position","engagement_rate",
             "scroll_rate","ai_traffic_pct","impression_tier","position_tier","y"}
assert not (set(SOURCES) & FORBIDDEN), "a forbidden column is feeding the features"

def make_features(d):
    X = pd.DataFrame(index=d.index)
    for a in ["imp", "clk", "ses"]:
        X["log_" + a + "_old"]  = np.log1p(d[a + "_old"])
        X["log_" + a + "_prev"] = np.log1p(d[a + "_prev"])
        X[a + "_momentum"] = X["log_" + a + "_prev"] - X["log_" + a + "_old"]   # >0 growing, <0 falling
    X["ctr_old"]  = d.clk_old  / d.imp_old.clip(lower=1)
    X["ctr_prev"] = d.clk_prev / d.imp_prev.clip(lower=1)
    X["ctr_delta"] = X.ctr_prev - X.ctr_old
    X["log_search_volume"] = np.log1p(d.search_volume)
    X["competition"], X["cpc"] = d.competition, d.cpc
    X["word_count"], X["wc_missing"] = d.word_count, d.word_count.isna().astype(int)
    X["content_age_days"] = d.content_age_days
    X["updated_over_30d_ago"] = (d.days_since_last_update > 30).astype(int)  # 0 = updated INSIDE the label window
    X["is_keyword_article"]  = (d.content_type == "keyword article").astype(int)
    X["is_comparison"]       = (d.content_type == "comparison article").astype(int)
    for v in ["informational", "transactional", "commercial", "navigational"]:
        X["intent_" + v] = (d.main_intent == v).astype(int)
    return X

X, y, g = make_features(el), el["y"].values, el["client_id"].values

def precision_at_k(y, s, k):
    return y[np.argsort(-s, kind="stable")[:k]].mean()

def cv_scores(model_fn, X, y, g, split="group", n=5, seed=0):
    """Out-of-fold probabilities. split='group' = client holdout (honest); 'random' = leaky comparison."""
    oof = np.zeros(len(y))
    sp = GroupKFold(n).split(X, y, g) if split == "group" else StratifiedKFold(n, shuffle=True, random_state=seed).split(X, y)
    for tr, te in sp:
        m = model_fn(); m.fit(X.iloc[tr], y[tr]); oof[te] = m.predict_proba(X.iloc[te])[:, 1]
    return oof

make_lr  = lambda: make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=500))
make_hgb = lambda: HGB(max_iter=200, learning_rate=0.05, max_depth=4, random_state=0)

def report(name, s, y=y):
    return {"model": name, "ROC-AUC": roc_auc_score(y, s), "PR-AUC": average_precision_score(y, s),
            "P@50": precision_at_k(y, s, 50), "P@100": precision_at_k(y, s, 100), "P@500": precision_at_k(y, s, 500)}

# Baselines (no fitting): A = the original ML-07 rule, B = "last period's drop continues"
score_rule = ((el.days_since_last_update >= 180) & (el.impressions_90d >= 500)).astype(float).values * el.impressions_90d.values
score_persist = (-X["imp_momentum"]).values

print(f"{X.shape[1]} features, {len(X):,} eligible pages")
print(list(X.columns))


all pages: 30,000 | eligible (prev30 impressions >= 100): 18,010 | clients: 30 | decline base rate: 0.616


25 features, 18,010 eligible pages
['log_imp_old', 'log_imp_prev', 'imp_momentum', 'log_clk_old', 'log_clk_prev', 'clk_momentum', 'log_ses_old', 'log_ses_prev', 'ses_momentum', 'ctr_old', 'ctr_prev', 'ctr_delta', 'log_search_volume', 'competition', 'cpc', 'word_count', 'wc_missing', 'content_age_days', 'updated_over_30d_ago', 'is_keyword_article', 'is_comparison', 'intent_informational', 'intent_transactional', 'intent_commercial', 'intent_navigational']


## 2. Split design

**Client-holdout (GroupKFold, 5 folds):** a client never appears in both train and test. This is the honest split because the real use is *scoring a client the model has never seen*, and pages from one client share traffic patterns, topics and site-level effects, so a random split quietly leaks client identity. One client holds about a third of the eligible pages, so fold 0 is a single large client; per-fold results vary and I report them pooled. A time-aware split is not possible on this snapshot (one row per page, trailing windows only); that limitation is stated in the paper, and the full warehouse would allow rolling-origin validation.

In [ ]:

folds = list(GroupKFold(5).split(X, y, g))
for i, (tr, te) in enumerate(folds):
    assert not set(g[tr]) & set(g[te])
    print(f"fold {i}: train {len(tr):>6,} pages / {len(set(g[tr]))} clients | test {len(te):>5,} pages / {len(set(g[te]))} clients | test base rate {y[te].mean():.3f}")


fold 0: train 12,320 pages / 29 clients | test 5,690 pages / 1 clients | test base rate 0.510
fold 1: train 14,658 pages / 29 clients | test 3,352 pages / 1 clients | test base rate 0.634
fold 2: train 15,020 pages / 23 clients | test 2,990 pages / 7 clients | test base rate 0.532
fold 3: train 15,021 pages / 21 clients | test 2,989 pages / 9 clients | test base rate 0.728
fold 4: train 15,021 pages / 18 clients | test 2,989 pages / 12 clients | test base rate 0.768


## 3. Train + compare vs my baseline

Same pages, same client-holdout folds, same metrics. Out-of-fold scores, so every page is scored by a model that never saw its client.

**Result:** both models beat both baselines on ranking quality (ROC-AUC ~0.61 vs 0.50-0.57), and logistic regression is as good as gradient boosting, so the simple model wins on explainability. The lift is real but modest: precision in the top 50 is ~0.88 vs a 0.62 base rate, and it fades as k grows (P@500 ~0.80).

In [ ]:

oof_lr  = cv_scores(make_lr,  X, y, g, "group")
oof_hgb = cv_scores(make_hgb, X, y, g, "group")
tbl = pd.DataFrame([report("random (reference)", np.random.RandomState(0).rand(len(y))),
                    report("A: stale>=180 & visible (original)", score_rule),
                    report("B: persistence baseline", score_persist),
                    report("Logistic regression (client holdout)", oof_lr),
                    report("Gradient boosting (client holdout)", oof_hgb)])
print(f"base rate {y.mean():.3f}\n"); print(tbl.round(3).to_string(index=False))


base rate 0.616

                               model  ROC-AUC  PR-AUC  P@50  P@100  P@500
                  random (reference)    0.495   0.610  0.46   0.53  0.590
  A: stale>=180 & visible (original)    0.501   0.616  0.76   0.65  0.604
             B: persistence baseline    0.567   0.665  0.66   0.64  0.636
Logistic regression (client holdout)    0.612   0.701  0.88   0.85  0.804
  Gradient boosting (client holdout)    0.615   0.692  0.82   0.74  0.696


## 4. Errors and interpretation

**What it leans on (logistic coefficients, standardized):** higher past impressions push risk *up*; higher click levels, informational/commercial/transactional intent, older pages and higher search volume push risk *down*. (Known-intent pages decline less than pages with no recorded intent, which are mostly `feedly` articles.) Content age pointing the opposite way from the stale-page folklore matches the signal audit.

**Where it is wrong:** the top-decile predicted probability (~0.88) overstates the real rate (~0.74), so the scores rank well but are **not calibrated probabilities**: use them for ordering, not as percentages. Per-client AUC for the 10 clients with 200+ pages ranges ~0.59-0.67, so the signal is modest but fairly consistent.

In [ ]:

m = make_lr(); m.fit(X, y)
coef = pd.Series(m[-1].coef_[0], index=X.columns).sort_values()
print("Top drivers (standardized logistic coefficients; + = more likely to decline):")
print(pd.concat([coef.head(6), coef.tail(6)]).round(3).to_string())

cal = pd.DataFrame({"p": oof_lr, "y": y}); cal["decile"] = pd.qcut(cal.p, 10, labels=False, duplicates="drop")
print("\nCalibration by predicted-risk decile:"); print(cal.groupby("decile").mean().round(3).to_string())

pc = [(c, len(d), roc_auc_score(d.y, d.s)) for c, d in el.assign(s=oof_lr).groupby("client_id") if len(d) >= 200 and d.y.nunique() > 1]
pc = pd.DataFrame(pc, columns=["client", "n_pages", "auc"]); print(f"\nPer-client AUC ({len(pc)} clients with 200+ pages): min {pc.auc.min():.2f}, median {pc.auc.median():.2f}, max {pc.auc.max():.2f}")
top = el.assign(s=oof_lr).sort_values("s", ascending=False).head(200)
print(f"\nTop-200 false positives: {(top.y == 0).sum()} pages; they come from {top[top.y == 0].client_id.nunique()} clients.")


Top drivers (standardized logistic coefficients; + = more likely to decline):
intent_informational   -0.357
content_age_days       -0.347
log_clk_prev           -0.344
log_clk_old            -0.286
intent_transactional   -0.271
intent_commercial      -0.261
ctr_prev                0.036
is_keyword_article      0.038
competition             0.048
ctr_delta               0.049
log_imp_old             0.236
log_imp_prev            0.267

Calibration by predicted-risk decile:
            p      y
decile              
0       0.363  0.475
1       0.468  0.466
2       0.525  0.529
3       0.578  0.572
4       0.626  0.626
5       0.667  0.675
6       0.707  0.672
7       0.749  0.694
8       0.800  0.708
9       0.882  0.741



Per-client AUC (10 clients with 200+ pages): min 0.59, median 0.63, max 0.67

Top-200 false positives: 33 pages; they come from 2 clients.


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.